# Clase 105 — Keras Tuner

El **hyperparameter tuning** busca sistemáticamente `n_layers`, `units`, `lr`,
`dropout`, etc. En vez de un grid manual usamos estrategias modernas: **Random
Search**, **Hyperband** y **Bayesian Optimization**, todas disponibles en
**Keras Tuner** (`keras_tuner`). Mencionamos también Optuna como alternativa industrial.

Requiere: `tensorflow` / `keras`, `keras-tuner` (`pip install keras-tuner`). Se ejecuta en Colab con GPU.

## 1. Datos y la `model-building function`

In [ ]:
import numpy as np
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.astype("float32") / 255.0
X_te = X_te.astype("float32") / 255.0

def build_model(hp):
    # los hiperparámetros se declaran con hp.Int / hp.Float / hp.Choice
    n_units = hp.Choice("units", values=[32, 64, 128])
    lr = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")  # log para LR
    modelo = keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(n_units, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

print("model-building function lista; hiperparámetros: units, lr")

## 2. `RandomSearch`: muestrear configs al azar

In [ ]:
tuner_rs = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=20,
    overwrite=True,
    directory="kt_random",
    project_name="fashion",
)
tuner_rs.search(X_tr, y_tr, validation_split=0.1, epochs=5, verbose=0)

mejor_hp = tuner_rs.get_best_hyperparameters(1)[0]
print("RandomSearch -> units:", mejor_hp.get("units"),
      "| lr:", round(mejor_hp.get("lr"), 5))

## 3. `Hyperband`: successive halving

Corre muchos trials cortos, mata los peores y sigue con los buenos por más épocas.

In [ ]:
tuner_hb = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=10,
    factor=3,               # cada ronda mantiene 1/factor de los trials
    overwrite=True,
    directory="kt_hyperband",
    project_name="fashion",
)
tuner_hb.search(X_tr, y_tr, validation_split=0.1, epochs=10, verbose=0)
hp_hb = tuner_hb.get_best_hyperparameters(1)[0]
print("Hyperband -> units:", hp_hb.get("units"), "| lr:", round(hp_hb.get("lr"), 5))

## 4. `BayesianOptimization`: modela el paisaje de búsqueda

In [ ]:
tuner_bo = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=15,
    overwrite=True,
    directory="kt_bayes",
    project_name="fashion",
)
tuner_bo.search(X_tr, y_tr, validation_split=0.1, epochs=5, verbose=0)
hp_bo = tuner_bo.get_best_hyperparameters(1)[0]
print("Bayesian -> units:", hp_bo.get("units"), "| lr:", round(hp_bo.get("lr"), 5))
print("Bayesian necesita >10-20 trials para superar claramente a Random.")

## 5. Reentrenar el mejor modelo y evaluar en test

In [ ]:
mejor_modelo = tuner_hb.hypermodel.build(hp_hb)   # modelo fresco con los mejores hp
early = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
mejor_modelo.fit(X_tr, y_tr, validation_split=0.1,
                 epochs=20, callbacks=[early], verbose=0)
test_loss, test_acc = mejor_modelo.evaluate(X_te, y_te, verbose=0)
print(f"accuracy en test del mejor modelo: {test_acc:.3f}")
print("El test set se usa UNA sola vez, al final, nunca durante la búsqueda.")

## 6. Espacio más rico: nº de capas y dropout

`hp.Int` con un loop permite variar la **profundidad** del modelo.

In [ ]:
def build_model_profundo(hp):
    modelo = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    for i in range(hp.Int("n_layers", 1, 4)):          # 1 a 4 capas ocultas
        modelo.add(layers.Dense(
            hp.Choice(f"units_{i}", [32, 64, 128, 256]), activation="relu"))
    modelo.add(layers.Dropout(hp.Float("dropout", 0.0, 0.5, step=0.1)))
    modelo.add(layers.Dense(10, activation="softmax"))
    lr = hp.Float("lr", 1e-5, 1e-2, sampling="log")
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

tuner = kt.Hyperband(build_model_profundo, objective="val_accuracy",
                     max_epochs=10, factor=3, overwrite=True,
                     directory="kt_profundo", project_name="fashion")
print("espacio: n_layers[1-4], units{32..256}, dropout[0-0.5], lr log[1e-5,1e-2]")

## Ejercicios

1. **RandomSearch**: tuneá `units ∈ {32,64,128}` y `lr ∈ [1e-4,1e-2]` log con 20
   trials y reportá los mejores hiperparámetros y la `val_accuracy`.
2. **Hyperband vs Random**: corré ambos con el mismo espacio y compará el tiempo
   total de búsqueda.
3. **`lr` log vs uniforme**: cambiá `sampling="log"` por lineal y observá cómo el
   tuner tarda mucho más en encontrar un buen learning rate.
4. **Migrar a Optuna**: reescribí el espacio con `trial.suggest_float('lr', 1e-5,
   1e-2, log=True)` y `optuna.create_study(direction='maximize')`.

## Conclusiones

- La **model-building function** declara el espacio con `hp.Int`, `hp.Float`, `hp.Choice`.
- **RandomSearch** casi siempre supera al grid search (Bergstra & Bengio, 2012).
- **Hyperband** ahorra cómputo con successive halving; **BayesianOptimization** aprende el paisaje.
- El **learning rate** se muestrea en escala **log** y se tunea primero: es el hiperparámetro más sensible.
- El **test set** queda intacto: se reporta una sola vez con el mejor modelo reentrenado.